In [ ]:
%%sql
-- Expected: "Database 'ZFACETS_DEV_CLONE' does not exist or not authorized."
-- Zero lag — enforced on the very next query, no session restart needed
USE ROLE BUSINESS_ANALYST_ROLE;
SELECT COUNT(*) AS visible_members FROM SILVER.MEMBER;

In [ ]:
%%sql
-- Revoke all access levels instantly
USE ROLE ACCOUNTADMIN;

REVOKE SELECT ON TABLE zFACETS_DEV_CLONE.SILVER.MEMBER  FROM ROLE BUSINESS_ANALYST_ROLE;
REVOKE USAGE ON SCHEMA zFACETS_DEV_CLONE.SILVER          FROM ROLE BUSINESS_ANALYST_ROLE;
REVOKE USAGE ON DATABASE zFACETS_DEV_CLONE               FROM ROLE BUSINESS_ANALYST_ROLE;

-- Verify grants are gone
SHOW GRANTS TO ROLE BUSINESS_ANALYST_ROLE;

In [ ]:
%%sql
-- Confirm BA currently has access (~30,593 COMM-only rows via row policy)
USE ROLE BUSINESS_ANALYST_ROLE;
SELECT COUNT(*) AS visible_members FROM SILVER.MEMBER;

## Part 2 — Instant REVOKE

HIPAA requires access can be removed immediately. We revoke all access levels (table → schema → database) and verify it takes effect on the very next query — no lag, no session invalidation needed.

In [ ]:
%%sql
-- Expected: Insufficient privileges to DROP
DROP TABLE SILVER.MEMBER;

In [ ]:
%%sql
-- Expected: Insufficient privileges to INSERT
INSERT INTO SILVER.MEMBER (MEME_ID, SBSB_ID, MEME_LAST_NAME, MEME_FIRST_NAME, MEME_MCTR_TYPE)
VALUES (99999, 99999, 'TEST', 'RECORD', 'COMM');

### Attempting to break out of the sandbox

The BA tries to INSERT and DROP on the governed `SILVER.MEMBER` table. Both fail — `SELECT` was explicitly granted but `INSERT` and DDL were not.

In [ ]:
%%sql
-- BA creates a view for reporting — works fine in their sandbox
CREATE OR REPLACE VIEW ANALYST.COMM_ACTIVE_MEMBERS AS
SELECT MEME_ID, MEME_MCTR_TYPE, MEME_REL_CD, MEMBER_STATUS
FROM SILVER.MEMBER
WHERE MEMBER_STATUS = 'Active';

SELECT COUNT(*) AS active_commercial_members FROM ANALYST.COMM_ACTIVE_MEMBERS;

In [ ]:
%%sql
-- Switch to Business Analyst role
USE ROLE BUSINESS_ANALYST_ROLE;

-- BA creates their own sandbox schema — they own it fully
CREATE SCHEMA IF NOT EXISTS ANALYST
    COMMENT = 'Business Analyst sandbox — read/write here, read-only on SILVER.';

-- CTAS: BA builds a working table from COMM members
-- Row access policy enforces they only see COMM rows — even in their own sandbox
CREATE OR REPLACE TABLE ANALYST.COMM_MEMBERS AS
SELECT
    MEME_ID,
    MEME_MCTR_TYPE,
    MEME_REL_CD,
    RELATIONSHIP_DESC,
    MEMBER_STATUS,
    ACTIVE_PCP_NAME
FROM SILVER.MEMBER;

SELECT * FROM ANALYST.COMM_MEMBERS LIMIT 10;
-- Note: only COMM rows — row access policy followed the data into the CTAS

## Part 1 — Separation of Duties

The `BUSINESS_ANALYST_ROLE` can create and own objects in their own `ANALYST` sandbox schema, but **cannot modify or drop governed SILVER tables**.

The row access policy travels with the data — even a CTAS into the BA's own schema only contains COMM rows.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES NONE;
USE DATABASE zFACETS_DEV_CLONE;
USE SCHEMA SILVER;
USE WAREHOUSE WH_XS;

# Security Demo — Part 1 & 2: Separation of Duties + Instant REVOKE

CalOptima RFP 26-038 | Topic 5 — Separation of Duties, REVOKE, Audit

**Part 1 — Separation of Duties**
The Business Analyst role has its own sandbox schema for analysis work. They cannot touch the governed SILVER tables.

**Part 2 — Instant REVOKE**
HIPAA requires that access can be removed immediately. We demonstrate zero-lag access revocation — no session invalidation, enforced on every query.